### 层与块

块，在构造自定义块之前，先回顾一下怎么用自带的nn.Sequential实现

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
net = nn.Sequential(nn.Linear(20,256),nn.ReLU(), nn.Linear(256,10))
X = torch.rand(3,20)
net(X) # net.__call__(X)的缩写，可以直接执行传播函数

tensor([[-0.0862,  0.0693, -0.1162, -0.0238,  0.0015, -0.0571,  0.0261, -0.0841,
          0.0670,  0.0230],
        [-0.0987,  0.0098, -0.1751, -0.1613, -0.0363, -0.0111, -0.0695,  0.0181,
         -0.0122, -0.0644],
        [-0.1958, -0.1035, -0.0744,  0.0531, -0.0951, -0.1124, -0.0619, -0.1909,
          0.1121, -0.0881]], grad_fn=<AddmmBackward0>)

层的执行顺序是作为参数传递的，nn.Sequential维护了由Module组成的有序列表

两个全连接层都是Linear的实例

这个前向传播函数将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入

In [ ]:
class MLP(nn.Module):
    # 模型参数声明层，这里我们声明两个全连接层
    def __init__(self):
        # 调用父类Module的初始化函数来必要的初始化，这样在类实例化时也可以指定别的参数
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))  # 先hidden，再relu，再out

In [13]:
net = MLP()
net(X)

tensor([[ 0.0654, -0.0458, -0.0947,  0.0653, -0.2166, -0.0903, -0.1728,  0.0696,
         -0.0117, -0.1129],
        [ 0.0563, -0.1492, -0.1418,  0.1001, -0.0835, -0.0116, -0.2047,  0.1149,
          0.0584, -0.1427],
        [ 0.1234, -0.0871, -0.1474, -0.0831, -0.1948,  0.0337, -0.1580,  0.0324,
          0.0157,  0.0140]], grad_fn=<AddmmBackward0>)

顺序块
需要定义两个关键函数：

一种将块逐个加到列表中的函数；

一种前向传播函数，用于将输入按追加块的顺序传递给块组成的链条

In [16]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module # _module的类型时orderdict
    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

In [17]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 20))
net(X)

tensor([[-0.1352,  0.1426, -0.0326, -0.1378,  0.1183, -0.1640,  0.1481, -0.1845,
          0.0263,  0.3483,  0.0086, -0.1525,  0.1198, -0.1203, -0.1299,  0.0503,
         -0.2082, -0.2358,  0.3186,  0.0794],
        [ 0.0730,  0.0893,  0.0381,  0.0360, -0.0731, -0.1824,  0.1922, -0.1139,
         -0.0317,  0.3662, -0.0792, -0.1126, -0.0296, -0.2089, -0.1607,  0.0341,
         -0.2134, -0.1536,  0.3957,  0.0318],
        [-0.0251,  0.0470, -0.0282, -0.1048,  0.0083, -0.1885,  0.0211, -0.0579,
         -0.0280,  0.2392, -0.0294, -0.1091,  0.0060, -0.2338, -0.0781,  0.0176,
         -0.1909, -0.1013,  0.2353,  0.0778]], grad_fn=<AddmmBackward0>)

如果用普通的列表，而不用orderdict

In [29]:
class MySequential1(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.bl = []
        for idx, m in enumerate(args):
            self.bl.append(m)
    
    def forward(self, X):
        for block in self.bl:
            X = block(X)
        return X

In [58]:
net = MySequential1(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 20))
net(X)
# print(net[1].state_dict())

tensor([[-0.0099, -0.0604, -0.0803,  0.1071, -0.1777,  0.0532,  0.0744, -0.0821,
         -0.0479,  0.0782,  0.0461,  0.0913,  0.1401, -0.0148, -0.1758, -0.0757,
          0.1376, -0.1960,  0.1441,  0.1030],
        [-0.0007, -0.0505, -0.0445,  0.0488, -0.1782,  0.0106, -0.0819, -0.0576,
         -0.0145,  0.0698,  0.0359,  0.0078,  0.0974,  0.1286, -0.1607,  0.0052,
          0.1296, -0.1524,  0.1941,  0.0007],
        [-0.1076, -0.0089, -0.0615,  0.1713, -0.1805,  0.0061, -0.0116, -0.0359,
         -0.0025,  0.1752,  0.1170, -0.0255,  0.1799,  0.0295, -0.1874,  0.0229,
          0.1578, -0.1900,  0.1185, -0.0643]], grad_fn=<AddmmBackward0>)

在前向传播函数中执行代码

In [ ]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20,20), requires_grad=False)
        self.linear = nn.Linear(20, 20)
    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X) # 复用全连接层。这相当于两个全连接层共享参数
        while X.abs().sum() > 1: # 执行我们需要的逻辑
            X /= 2
        return X.abs().sum()

In [45]:
net = FixedHiddenMLP()
net(X)

tensor(0.5998, grad_fn=<SumBackward0>)

混合搭配组合块

In [49]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)
    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)


tensor(0.8644, grad_fn=<SumBackward0>)

In [53]:
class PMPL1(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))  # 先hidden，再relu，再out
    
class PMPL2(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(10, 256)
        self.output = nn.Linear(256, 20)
    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))  # 先hidden，再relu，再out

class ParallerMLP(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2
    def forward(self, X):
        pmpl1 = self.net1()
        pmpl2 = self.net2()
        X = pmpl1(X)
        X = pmpl2(X)
        return X

parnet = ParallerMLP(PMPL1,PMPL2)
parnet(X)       

tensor([[-7.1413e-02, -3.4688e-02,  1.1177e-01,  1.1232e-01, -3.7961e-02,
         -1.4884e-01,  3.0932e-02, -1.0127e-01,  7.1531e-02,  1.3204e-02,
          5.1400e-02,  1.7620e-02, -1.9414e-02, -2.4954e-02,  4.8306e-02,
          2.9852e-02, -7.1433e-02,  3.0474e-02, -7.3710e-02,  3.4978e-02],
        [-3.6647e-02, -3.1103e-02,  1.1975e-01,  9.4366e-02, -3.8415e-02,
         -1.5131e-01,  2.5494e-02, -7.3312e-02,  8.5471e-02,  3.4098e-03,
          4.4314e-02,  3.8692e-03,  1.2084e-02, -1.5273e-02,  3.5809e-02,
          1.7297e-05, -7.4243e-02,  3.8184e-02, -9.5779e-02,  3.0458e-02],
        [-5.8447e-02, -3.0578e-02,  1.0993e-01,  1.4666e-01, -3.5515e-02,
         -1.5797e-01,  2.7435e-02, -7.6945e-02,  8.6590e-02,  4.7143e-02,
          1.4770e-02,  3.2651e-02,  5.7115e-03, -5.5368e-02,  4.5711e-02,
          1.0798e-02, -3.5756e-02,  3.9832e-02, -6.5031e-02,  1.0843e-02]],
       grad_fn=<AddmmBackward0>)